# newsrec_bench — Kaggle runner (V10)

Code nằm trong `training/` của repo; notebook này chỉ clone repo, import các script và chạy.

1. Settings: bật GPU + Internet. Add Input: dataset có `MINDsmall_train` và `MINDsmall_dev`.
2. Sửa `REPO_URL` / `BRANCH` bên dưới. Repo private → thêm Kaggle Secret `GITHUB_TOKEN`.
3. Run All. Artifact nằm ở `/kaggle/working/{news_emb.npz,v10_final}` — tải về và đặt vào `inference/artifacts/`.

In [ ]:
REPO_URL = 'https://github.com/nguyenpnguyen/recsys-project.git'
BRANCH = 'main'
MODELS = ['nrms', 'naml', 'fastformer', 'caum', 'lightgcn', 'llmenc_ca',
          'supermodel', 'super_no_diff', 'super_no_ca', 'super_no_graph', 'super_no_cl', 'super_no_bge',
          'popularity', 'bge_zeroshot', 'bge_zs_pop']
TRAIN_ARGS = ['--max-epochs', '12', '--min-epochs', '3', '--patience', '2', '--min-delta', '0.001',
              '--dim', '64', '--seed', '0', '--batch-size', '64', '--eval-batch', '32',
              '--lr', '1e-3', '--weight-decay', '1e-5', '--dev-ratio', '0.10']
BGE_MODEL = 'BAAI/bge-small-en-v1.5'

In [ ]:
import os, sys, glob, subprocess
from pathlib import Path

ON_KAGGLE = os.path.isdir('/kaggle/working')
if ON_KAGGLE:
    REPO = Path('/kaggle/working/repo'); WORK = Path('/kaggle/working')
    if not REPO.exists():
        url = REPO_URL
        try:
            from kaggle_secrets import UserSecretsClient
            url = url.replace('https://', f"https://{UserSecretsClient().get_secret('GITHUB_TOKEN')}@")
        except Exception:
            pass  # public repo
        subprocess.run(['git', 'clone', '--depth', '1', '-b', BRANCH, url, str(REPO)], check=True)
    hits = glob.glob('/kaggle/input/**/MINDsmall_train/news.tsv', recursive=True)
    assert hits, 'Add Input: dataset chứa MINDsmall_train/ và MINDsmall_dev/'
    DATA = Path(hits[0]).parent.parent
else:  # chạy local từ training/
    REPO = Path.cwd().parent; WORK = REPO / 'inference' / 'artifacts'; DATA = REPO / 'dataset'

sys.path.insert(0, str(REPO / 'training'))
MIND_TRAIN, MIND_DEV = DATA / 'MINDsmall_train', DATA / 'MINDsmall_dev'
NEWS_EMB, OUT = WORK / 'news_emb.npz', WORK / 'v10_final'
assert (MIND_DEV / 'behaviors.tsv').exists(), MIND_DEV

import torch
print('torch', torch.__version__, '| GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')
print('DATA =', DATA, '| OUT =', OUT)

### 1. BGE embedding (title + abstract, union train + dev)

In [ ]:
import numpy as np, llm_embed
if not (NEWS_EMB.exists() and str(np.load(NEWS_EMB)['model_name']) == BGE_MODEL):
    llm_embed.main(['--news', str(MIND_TRAIN / 'news.tsv'), str(MIND_DEV / 'news.tsv'),
                    '--out', str(NEWS_EMB), '--model', BGE_MODEL])
print('NEWS_EMB =', NEWS_EMB)

### 2. Train + test 12 model

In [ ]:
import bench
bench.main(['--mind-train', str(MIND_TRAIN), '--mind-dev', str(MIND_DEV), '--news-emb', str(NEWS_EMB),
            '--out', str(OUT), '--models', *MODELS, *TRAIN_ARGS])

### 3. Kiểm tra artifact

In [ ]:
bench.verify(OUT)